<a href="https://colab.research.google.com/github/serahnjogu-new/Climate-and-health-risk-prediction/blob/main/Climate_and_health_risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import warnings
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from google.colab import files

warnings.filterwarnings('ignore')

# 1. LOAD DATA
train   = pd.read_csv('/content/Train.csv')
test    = pd.read_csv('/content/Test.csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission.csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. OPTIMIZED FEATURE ENGINEERING (The 0.838 Formula + Temperature Anomalies)
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['year']  = df['deathdate'].dt.year
    df['month'] = df['deathdate'].dt.month

    # Original Season Mapping (Linear works better here than cyclical)
    df['season'] = df['month'].map({
        12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
        6:2, 7:2, 8:2, 9:3, 10:3, 11:3
    })

    # Granular Age Bins from your 0.838 version
    df['log_age']      = np.log1p(df['age'])
    df['age_squared']  = df['age'] ** 2
    df['is_infant']    = (df['age'] == 0).astype(int)
    df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
    df['is_child']     = (df['age'].between(1, 5)).astype(int)
    df['is_elderly']   = (df['age'] >= 60).astype(int)
    df['age_group']    = pd.cut(df['age'], bins=[-1,0,1,2,5,10,15,30,60,200],
                                labels=[0,1,2,3,4,5,6,7,8]).astype(int)

    # Temperature Anomaly (The "Climate" Signal)
    df['year_avg_temp'] = df.groupby('year')['max_temperature'].transform('mean')
    df['temp_anomaly']  = df['max_temperature'] - df['year_avg_temp']

    # Year Sensitivity logic
    year_sensitivity = {
        2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
        2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
        2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
    }
    df['year_sensitivity'] = df['year'].map(year_sensitivity)
    df['year_sens_age']    = df['year_sensitivity'] * df['age']

    # Simple Categorical mapping
    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp']
    return df.drop(drop_cols, axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.fillna(-999)

# 3. K-FOLD ENSEMBLE (LGBM + XGB)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()

oof_lgbm, oof_xgb = np.zeros(len(X)), np.zeros(len(X))
test_lgbm, test_xgb = np.zeros(len(X_test)), np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Using 0.838 winning params
    lgbm = LGBMClassifier(n_estimators=2000, learning_rate=0.03, max_depth=6,
                          scale_pos_weight=2.5, colsample_bytree=0.8, subsample=0.8, random_state=42)
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(100), log_evaluation(0)])

    xgb = XGBClassifier(n_estimators=2000, learning_rate=0.03, max_depth=6,
                        scale_pos_weight=neg/pos, colsample_bytree=0.8, subsample=0.8,
                        random_state=42, early_stopping_rounds=100, eval_metric='auc')
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    print(f"Fold {fold+1} complete.")

# 4. BLENDING & THRESHOLD OPTIMIZATION
# 60/40 blend often works better if one model (usually LGBM) has a higher AUC
oof_ensemble = (0.6 * oof_lgbm) + (0.4 * oof_xgb)
test_ensemble = (0.6 * test_lgbm) + (0.4 * test_xgb)

best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.3, 0.7, 0.01):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\nFinal OOF Score: {best_score:.4f} at Threshold: {best_thresh:.2f}")

# 5. SUBMISSION
submission = pd.DataFrame({
    'ID': ss['ID'],
    'TargetF1': (test_ensemble >= best_thresh).astype(int),
    'TargetRAUC': test_ensemble
})
submission.to_csv('refined_winning_submission.csv', index=False)
files.download('refined_winning_submission.csv')

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings('ignore')

# 1. LOAD DATA
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. OPTIMIZED FEATURE ENGINEERING (Retaining Lat/Lon & 6th-Place Logic)
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['year']  = df['deathdate'].dt.year
    df['month'] = df['deathdate'].dt.month
    df['day']   = df['deathdate'].dt.day
    df['dayofweek'] = df['deathdate'].dt.dayofweek

    # Season Mapping
    df['season'] = df['month'].map({
        12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
        6:2, 7:2, 8:2, 9:3, 10:3, 11:3
    })

    # Granular Age Bins
    df['log_age']   = np.log1p(df['age'])
    df['age_squared']  = df['age'] ** 2
    df['is_infant']    = (df['age'] == 0).astype(int)
    df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
    df['is_child']     = (df['age'].between(1, 5)).astype(int)
    df['is_elderly']   = (df['age'] >= 60).astype(int)
    df['age_group']    = pd.cut(df['age'], bins=[-1,0,1,2,5,10,15,30,60,200],
                                labels=[0,1,2,3,4,5,6,7,8]).astype(int)

    # Temperature Anomaly
    df['year_avg_temp'] = df.groupby('year')['max_temperature'].transform('mean')
    df['temp_anomaly']  = df['max_temperature'] - df['year_avg_temp']

    # Year Sensitivity logic (with safe fallback for recent years)
    year_sensitivity = {
        2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
        2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
        2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661, 2023: 0.600, 2024: 0.600,
        2025: 0.600, 2026: 0.600
    }
    df['year_sensitivity'] = df['year'].map(year_sensitivity).fillna(0.6)
    df['year_sens_age']    = df['year_sensitivity'] * df['age']

    # Categorical mapping (keeping lat and lon active)
    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp']
    return df.drop(drop_cols, axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X = train_df.drop('is_climate_sensitive', axis=1)
y = train_df['is_climate_sensitive']
X_test = test_df

# Handle any remaining categorical columns
for c in X.select_dtypes(include=['object', 'category']).columns:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X[c] = oe.fit_transform(X[[c]])
    X_test[c] = oe.transform(X_test[[c]])

X = X.fillna(-999)
X_test = X_test.fillna(-999)

# 3. K-FOLD ENSEMBLE (HistGradientBoosting + RandomForest)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_hgb, oof_rf = np.zeros(len(X)), np.zeros(len(X))
test_hgb, test_rf = np.zeros(len(X_test)), np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Model 1: HistGradientBoosting
    hgb = HistGradientBoostingClassifier(random_state=42, class_weight='balanced', max_iter=200)
    hgb.fit(X_tr, y_tr)

    # Model 2: RandomForest
    rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced', max_depth=10)
    rf.fit(X_tr, y_tr)

    oof_hgb[val_idx] = hgb.predict_proba(X_val)[:, 1]
    oof_rf[val_idx]  = rf.predict_proba(X_val)[:, 1]

    test_hgb += hgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_rf  += rf.predict_proba(X_test)[:, 1] / folds.n_splits
    print(f"Fold {fold+1} complete.")

# 4. BLENDING & THRESHOLD OPTIMIZATION
oof_ensemble = (0.5 * oof_hgb) + (0.5 * oof_rf)
test_ensemble = (0.5 * test_hgb) + (0.5 * test_rf)

best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.01):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

final_f1 = f1_score(y, (oof_ensemble >= best_thresh).astype(int))
final_auc = roc_auc_score(y, oof_ensemble)
print(f"\nValidation F1: {final_f1:.4f} | Validation AUC: {final_auc:.4f}")
print(f"Final OOF Score: {best_score:.4f} at Threshold: {best_thresh:.2f}")

# 5. SUBMISSION
submission = pd.DataFrame({
    'ID': ss['ID'],
    'TargetF1': (test_ensemble >= best_thresh).astype(int),
    'TargetRAUC': test_ensemble
})
submission.to_csv('refined_winning_submission_v2.csv', index=False)
print("Submission file saved successfully!")

Fold 1 complete.
Fold 2 complete.
Fold 3 complete.
Fold 4 complete.
Fold 5 complete.

Validation F1: 0.8104 | Validation AUC: 0.8067
Final OOF Score: 0.8089 at Threshold: 0.25
Submission file saved successfully!


In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings('ignore')

# 1. LOAD DATA
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. FEATURE ENGINEERING (Dropping Lat/Lon for cleaner generalization)
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['year']  = df['deathdate'].dt.year
    df['month'] = df['deathdate'].dt.month

    # Season Mapping
    df['season'] = df['month'].map({
        12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
        6:2, 7:2, 8:2, 9:3, 10:3, 11:3
    })

    # Age Bins & Interactions
    df['log_age']   = np.log1p(df['age'])
    df['age_squared']  = df['age'] ** 2
    df['is_infant']    = (df['age'] == 0).astype(int)
    df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
    df['is_child']     = (df['age'].between(1, 5)).astype(int)
    df['is_elderly']   = (df['age'] >= 60).astype(int)
    df['age_group']    = pd.cut(df['age'], bins=[-1,0,1,2,5,10,15,30,60,200],
                                labels=[0,1,2,3,4,5,6,7,8]).astype(int)

    # Temperature Anomaly
    df['year_avg_temp'] = df.groupby('year')['max_temperature'].transform('mean')
    df['temp_anomaly']  = df['max_temperature'] - df['year_avg_temp']

    # Year Sensitivity
    year_sensitivity = {
        2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
        2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
        2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661, 2023: 0.600, 2024: 0.600,
        2025: 0.600, 2026: 0.600
    }
    df['year_sensitivity'] = df['year'].map(year_sensitivity).fillna(0.6)
    df['year_sens_age']    = df['year_sensitivity'] * df['age']

    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    # Drop spatial and duplicate columns that cause noise
    drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp', 'latitude', 'longitude']
    return df.drop(drop_cols, axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X = train_df.drop('is_climate_sensitive', axis=1)
y = train_df['is_climate_sensitive']
X_test = test_df

for c in X.select_dtypes(include=['object', 'category']).columns:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X[c] = oe.fit_transform(X[[c]])
    X_test[c] = oe.transform(X_test[[c]])

X = X.fillna(-999)
X_test = X_test.fillna(-999)

# 3. K-FOLD ENSEMBLE (ExtraTrees + HistGradientBoosting)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_et, oof_hgb = np.zeros(len(X)), np.zeros(len(X))
test_et, test_hgb = np.zeros(len(X_test)), np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Model 1: ExtraTrees
    et = ExtraTreesClassifier(n_estimators=300, max_depth=12, random_state=42, class_weight='balanced', n_jobs=-1)
    et.fit(X_tr, y_tr)

    # Model 2: HistGradientBoosting
    hgb = HistGradientBoostingClassifier(random_state=42, class_weight='balanced', max_iter=250, learning_rate=0.03)
    hgb.fit(X_tr, y_tr)

    oof_et[val_idx]  = et.predict_proba(X_val)[:, 1]
    oof_hgb[val_idx] = hgb.predict_proba(X_val)[:, 1]

    test_et  += et.predict_proba(X_test)[:, 1] / folds.n_splits
    test_hgb += hgb.predict_proba(X_test)[:, 1] / folds.n_splits

# 4. OPTIMIZED BLENDING & THRESHOLDING
oof_ensemble = (0.6 * oof_et) + (0.4 * oof_hgb)
test_ensemble = (0.6 * test_et) + (0.4 * test_hgb)

best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.01):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

final_f1 = f1_score(y, (oof_ensemble >= best_thresh).astype(int))
final_auc = roc_auc_score(y, oof_ensemble)
print(f"\nNew Local OOF F1: {final_f1:.4f} | AUC: {final_auc:.4f} | Official Score: {best_score:.4f}")
print(f"Optimal Threshold: {best_thresh:.2f}")

# 5. SUBMISSION EXPORT
submission = pd.DataFrame({
    'ID': ss['ID'],
    'TargetF1': (test_ensemble >= best_thresh).astype(int),
    'TargetRAUC': test_ensemble
})
submission.to_csv('extratrees_winning_submission.csv', index=False)
print("Saved extratrees_winning_submission.csv successfully!")


New Local OOF F1: 0.8168 | AUC: 0.8162 | Official Score: 0.8166
Optimal Threshold: 0.30
Saved extratrees_winning_submission.csv successfully!


In [ ]:
!pip install catboost lightgbm xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb

warnings.filterwarnings('ignore')

# 1. LOAD DATA
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. FEATURE ENGINEERING
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['year']  = df['deathdate'].dt.year
    df['month'] = df['deathdate'].dt.month

    df['season'] = df['month'].map({
        12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
        6:2, 7:2, 8:2, 9:3, 10:3, 11:3
    })

    df['log_age']   = np.log1p(df['age'])
    df['age_squared']  = df['age'] ** 2
    df['is_infant']    = (df['age'] == 0).astype(int)
    df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
    df['is_child']     = (df['age'].between(1, 5)).astype(int)
    df['is_elderly']   = (df['age'] >= 60).astype(int)
    df['age_group']    = pd.cut(df['age'], bins=[-1,0,1,2,5,10,15,30,60,200],
                                labels=[0,1,2,3,4,5,6,7,8]).astype(int)

    df['year_avg_temp'] = df.groupby('year')['max_temperature'].transform('mean')
    df['temp_anomaly']  = df['max_temperature'] - df['year_avg_temp']

    year_sensitivity = {
        2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
        2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
        2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661, 2023: 0.600, 2024: 0.600,
        2025: 0.600, 2026: 0.600
    }
    df['year_sensitivity'] = df['year'].map(year_sensitivity).fillna(0.6)
    df['year_sens_age']    = df['year_sensitivity'] * df['age']

    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp', 'latitude', 'longitude']
    return df.drop(drop_cols, axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X = train_df.drop('is_climate_sensitive', axis=1)
y = train_df['is_climate_sensitive']
X_test = test_df

# Ordinal Encoding for Tree models
X_encoded = X.copy()
X_test_encoded = X_test.copy()
for c in X_encoded.select_dtypes(include=['object', 'category']).columns:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_encoded[c] = oe.fit_transform(X_encoded[[c]])
    X_test_encoded[c] = oe.transform(X_test_encoded[[c]])

X_encoded = X_encoded.fillna(-999)
X_test_encoded = X_test_encoded.fillna(-999)

# 3. 5-FOLD MULTI-MODEL TRAINING
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X))
oof_cb  = np.zeros(len(X))
oof_et  = np.zeros(len(X))

test_lgb = np.zeros(len(X_test))
test_cb  = np.zeros(len(X_test))
test_et  = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    print(f"Training Fold {fold+1}...")

    # Split Data
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    X_tr_enc, X_val_enc = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]

    # --- LightGBM ---
    clf_lgb = lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.03, max_depth=6,
        class_weight='balanced', random_state=42, verbose=-1
    )
    clf_lgb.fit(X_tr_enc, y_tr)
    oof_lgb[val_idx] = clf_lgb.predict_proba(X_val_enc)[:, 1]
    test_lgb += clf_lgb.predict_proba(X_test_encoded)[:, 1] / folds.n_splits

    # --- CatBoost ---
    clf_cb = CatBoostClassifier(
        iterations=500, learning_rate=0.03, depth=6,
        auto_class_weights='Balanced', verbose=0, random_seed=42
    )
    clf_cb.fit(X_tr, y_tr) # CatBoost handles raw categoricals well if desired, or encoded
    oof_cb[val_idx] = clf_cb.predict_proba(X_val)[:, 1]
    test_cb += clf_cb.predict_proba(X_test)[:, 1] / folds.n_splits

    # --- ExtraTrees ---
    clf_et = ExtraTreesClassifier(
        n_estimators=300, max_depth=12, class_weight='balanced',
        random_state=42, n_jobs=-1
    )
    clf_et.fit(X_tr_enc, y_tr)
    oof_et[val_idx] = clf_et.predict_proba(X_val_enc)[:, 1]
    test_et += clf_et.predict_proba(X_test_encoded)[:, 1] / folds.n_splits

# 4. OPTIMAL WEIGHT BLENDING
# Blend weights: 40% CatBoost, 40% LightGBM, 20% ExtraTrees
oof_blend = (0.4 * oof_cb) + (0.4 * oof_lgb) + (0.2 * oof_et)
test_blend = (0.4 * test_cb) + (0.4 * test_lgb) + (0.2 * test_et)

best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.01):
    f1 = f1_score(y, (oof_blend >= thresh).astype(int))
    auc = roc_auc_score(y, oof_blend)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

final_f1 = f1_score(y, (oof_blend >= best_thresh).astype(int))
final_auc = roc_auc_score(y, oof_blend)
print(f"\nFinal Blend OOF F1: {final_f1:.4f} | AUC: {final_auc:.4f} | Official Score: {best_score:.4f}")
print(f"Optimal Threshold: {best_thresh:.2f}")

# 5. SUBMISSION EXPORT
submission = pd.DataFrame({
    'ID': ss['ID'],
    'TargetF1': (test_blend >= best_thresh).astype(int),
    'TargetRAUC': test_blend
})
submission.to_csv('catboost_lgbm_ensemble_submission.csv', index=False)
print("Saved catboost_lgbm_ensemble_submission.csv successfully!")

Training Fold 1...
Training Fold 2...
Training Fold 3...
Training Fold 4...
Training Fold 5...

Final Blend OOF F1: 0.8182 | AUC: 0.8144 | Official Score: 0.8167
Optimal Threshold: 0.25
Saved catboost_lgbm_ensemble_submission.csv successfully!


In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings('ignore')

# 1. LOAD DATA
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. FEATURE ENGINEERING (Retaining Latitude and Longitude with strong regularization)
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['year']  = df['deathdate'].dt.year
    df['month'] = df['deathdate'].dt.month

    # Season Mapping
    df['season'] = df['month'].map({
        12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
        6:2, 7:2, 8:2, 9:3, 10:3, 11:3
    })

    # Age Bins & Interactions
    df['log_age']   = np.log1p(df['age'])
    df['age_squared']  = df['age'] ** 2
    df['is_infant']    = (df['age'] == 0).astype(int)
    df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
    df['is_child']     = (df['age'].between(1, 5)).astype(int)
    df['is_elderly']   = (df['age'] >= 60).astype(int)
    df['age_group']    = pd.cut(df['age'], bins=[-1,0,1,2,5,10,15,30,60,200],
                                labels=[0,1,2,3,4,5,6,7,8]).astype(int)

    # Temperature Anomaly
    df['year_avg_temp'] = df.groupby('year')['max_temperature'].transform('mean')
    df['temp_anomaly']  = df['max_temperature'] - df['year_avg_temp']

    # Year Sensitivity
    year_sensitivity = {
        2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
        2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
        2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661, 2023: 0.600, 2024: 0.600,
        2025: 0.600, 2026: 0.600
    }
    df['year_sensitivity'] = df['year'].map(year_sensitivity).fillna(0.6)
    df['year_sens_age']    = df['year_sensitivity'] * df['age']

    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    # Keep latitude and longitude active while dropping unnecessary ID/metadata columns
    drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp']
    return df.drop(drop_cols, axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X = train_df.drop('is_climate_sensitive', axis=1)
y = train_df['is_climate_sensitive']
X_test = test_df

for c in X.select_dtypes(include=['object', 'category']).columns:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X[c] = oe.fit_transform(X[[c]])
    X_test[c] = oe.transform(X_test[[c]])

X = X.fillna(-999)
X_test = X_test.fillna(-999)

# 3. K-FOLD ENSEMBLE WITH REGULARIZED SPATIAL TREES
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_hgb, oof_rf = np.zeros(len(X)), np.zeros(len(X))
test_hgb, test_rf = np.zeros(len(X_test)), np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Model 1: HistGradientBoosting (regularized to prevent lat/lon overfitting)
    hgb = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.02, max_leaf_nodes=31, min_samples_leaf=30,
        l2_regularization=1.0, class_weight='balanced', random_state=42
    )
    hgb.fit(X_tr, y_tr)

    # Model 2: RandomForest (constrained depth to handle spatial coordinates smoothly)
    rf = RandomForestClassifier(
        n_estimators=400, max_depth=10, min_samples_split=10, min_samples_leaf=5,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    rf.fit(X_tr, y_tr)

    oof_hgb[val_idx] = hgb.predict_proba(X_val)[:, 1]
    oof_rf[val_idx]  = rf.predict_proba(X_val)[:, 1]

    test_hgb += hgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_rf  += rf.predict_proba(X_test)[:, 1] / folds.n_splits

# 4. BLENDING & STABLE THRESHOLDING
oof_ensemble = (0.5 * oof_hgb) + (0.5 * oof_rf)
test_ensemble = (0.5 * test_hgb) + (0.5 * test_rf)

# Use a stable fixed threshold (0.45) to avoid OOF threshold optimization overfitting
best_thresh = 0.45
final_f1 = f1_score(y, (oof_ensemble >= best_thresh).astype(int))
final_auc = roc_auc_score(y, oof_ensemble)
final_score = (0.6 * final_f1) + (0.4 * final_auc)

print(f"\nValidation F1: {final_f1:.4f} | Validation AUC: {final_auc:.4f} | Official Score: {final_score:.4f}")

# 5. SUBMISSION EXPORT
submission = pd.DataFrame({
    'ID': ss['ID'],
    'TargetF1': (test_ensemble >= best_thresh).astype(int),
    'TargetRAUC': test_ensemble
})
submission.to_csv('spatial_regularized_submission.csv', index=False)
print("Saved spatial_regularized_submission.csv successfully!")


Validation F1: 0.7900 | Validation AUC: 0.8166 | Official Score: 0.8007
Saved spatial_regularized_submission.csv successfully!


In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings('ignore')

# 1. LOAD DATA
train   = pd.read_csv('Train (6).csv')
test    = pd.read_csv('Test (7).csv')
climate = pd.read_csv('climate_features.csv')
ss      = pd.read_csv('SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. FEATURE ENGINEERING (Retaining Latitude and Longitude with strong regularization)
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['year']  = df['deathdate'].dt.year
    df['month'] = df['deathdate'].dt.month

    # Season Mapping
    df['season'] = df['month'].map({
        12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
        6:2, 7:2, 8:2, 9:3, 10:3, 11:3
    })

    # Age Bins & Interactions
    df['log_age']   = np.log1p(df['age'])
    df['age_squared']  = df['age'] ** 2
    df['is_infant']    = (df['age'] == 0).astype(int)
    df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
    df['is_child']     = (df['age'].between(1, 5)).astype(int)
    df['is_elderly']   = (df['age'] >= 60).astype(int)
    df['age_group']    = pd.cut(df['age'], bins=[-1,0,1,2,5,10,15,30,60,200],
                                labels=[0,1,2,3,4,5,6,7,8]).astype(int)

    # Temperature Anomaly
    df['year_avg_temp'] = df.groupby('year')['max_temperature'].transform('mean')
    df['temp_anomaly']  = df['max_temperature'] - df['year_avg_temp']

    # Year Sensitivity
    year_sensitivity = {
        2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
        2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
        2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661, 2023: 0.600, 2024: 0.600,
        2025: 0.600, 2026: 0.600
    }
    df['year_sensitivity'] = df['year'].map(year_sensitivity).fillna(0.6)
    df['year_sens_age']    = df['year_sensitivity'] * df['age']

    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp']
    return df.drop(drop_cols, axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X = train_df.drop('is_climate_sensitive', axis=1)
y = train_df['is_climate_sensitive']
X_test = test_df

for c in X.select_dtypes(include=['object', 'category']).columns:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X[c] = oe.fit_transform(X[[c]])
    X_test[c] = oe.transform(X_test[[c]])

X = X.fillna(-999)
X_test = X_test.fillna(-999)

# 3. K-FOLD ENSEMBLE WITH REGULARIZED SPATIAL TREES
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_hgb, oof_rf = np.zeros(len(X)), np.zeros(len(X))
test_hgb, test_rf = np.zeros(len(X_test)), np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    hgb = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.02, max_leaf_nodes=31, min_samples_leaf=30,
        l2_regularization=1.0, class_weight='balanced', random_state=42
    )
    hgb.fit(X_tr, y_tr)

    rf = RandomForestClassifier(
        n_estimators=400, max_depth=10, min_samples_split=10, min_samples_leaf=5,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    rf.fit(X_tr, y_tr)

    oof_hgb[val_idx] = hgb.predict_proba(X_val)[:, 1]
    oof_rf[val_idx]  = rf.predict_proba(X_val)[:, 1]

    test_hgb += hgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_rf  += rf.predict_proba(X_test)[:, 1] / folds.n_splits

oof_blend = (0.5 * oof_hgb) + (0.5 * oof_rf)
test_blend = (0.5 * test_hgb) + (0.5 * test_rf)

# STRICT RULE COMPLIANCE: Default 0.5 threshold for TargetF1
DEFAULT_THRESH = 0.5
final_f1 = f1_score(y, (oof_blend >= DEFAULT_THRESH).astype(int))
final_auc = roc_auc_score(y, oof_blend)
final_score = (0.6 * final_f1) + (0.4 * final_auc)

print(f"Validation F1 (at 0.5): {final_f1:.4f} | Validation AUC: {final_auc:.4f} | Official Score: {final_score:.4f}")

# 4. SUBMISSION EXPORT WITH STRICT COLUMNS (ID, TargetF1, TargetRAUC)
submission = pd.DataFrame({
    'ID': ss['ID'],
    'TargetF1': (test_blend >= DEFAULT_THRESH).astype(int),
    'TargetRAUC': test_blend
})
submission.to_csv('compliant_submission.csv', index=False)
print("Saved compliant_submission.csv successfully!")

Validation F1 (at 0.5): 0.7750 | Validation AUC: 0.8166 | Official Score: 0.7917
Saved compliant_submission.csv successfully!


In [ ]:
import pandas as pd
import numpy as np
import warnings
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 1. LOAD DATA
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. Feature Engineering
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['year']  = df['deathdate'].dt.year
    df['month'] = df['deathdate'].dt.month

    # Season mapping
    df['season'] = df['month'].map({
        12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
        6:2, 7:2, 8:2, 9:3, 10:3, 11:3
    })

    # Age features & brackets
    df['log_age']      = np.log1p(df['age'])
    df['age_squared']  = df['age'] ** 2
    df['is_infant']    = (df['age'] == 0).astype(int)
    df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
    df['is_child']     = (df['age'].between(1, 5)).astype(int)
    df['is_elderly']   = (df['age'] >= 60).astype(int)
    df['age_group']    = pd.cut(df['age'], bins=[-1,0,1,2,5,10,15,30,60,200],
                                labels=[0,1,2,3,4,5,6,7,8]).astype(int)

    # Temperature anomaly features
    df['year_avg_temp'] = df.groupby('year')['max_temperature'].transform('mean')
    df['temp_anomaly']  = df['max_temperature'] - df['year_avg_temp']

    # Historical year sensitivity mapping
    year_sensitivity = {
        2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
        2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
        2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
    }
    df['year_sensitivity'] = df['year'].map(year_sensitivity)
    df['year_sens_age']    = df['year_sensitivity'] * df['age']

    # Categorical encoding
    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp']
    return df.drop(drop_cols, axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.fillna(-999)

# 3. Cross-Validation & Model Training
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))

print("Training models across 5 folds...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # LightGBM
    lgbm = LGBMClassifier(
        n_estimators=2500, learning_rate=0.02, max_depth=6, num_leaves=31,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.7,
        subsample=0.8, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(150), log_evaluation(0)])

    # XGBoost
    xgb = XGBClassifier(
        n_estimators=2500, learning_rate=0.02, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.7,
        subsample=0.8, random_state=42, early_stopping_rounds=150, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    # CatBoost
    cat = CatBoostClassifier(
        iterations=2500, learning_rate=0.03, depth=6,
        scale_pos_weight=scale_pos_weight_val, random_seed=42,
        early_stopping_rounds=150, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    # OOF predictions
    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]

    # Test predictions
    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

# 4. Weighted Blending
# Blending weights: 40% LightGBM, 30% XGBoost, 30% CatBoost
oof_ensemble = (0.4 * oof_lgbm) + (0.3 * oof_xgb) + (0.3 * oof_cat)
test_ensemble = (0.4 * test_lgbm) + (0.3 * test_xgb) + (0.3 * test_cat)

# 5. Expanded Threshold Optimization for Competition Metric (0.6*F1 + 0.4*AUC)
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.01):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Optimized Results ---")
print(f"Best Threshold: {best_thresh:.2f}")
print(f"OOF Competition Score: {best_score:.4f}")

print(f"\n--- Results ---")
print(f"Optimal Threshold: {best_thresh:.2f}")
print(f"OOF Competition Score: {best_score:.4f}")
print(f"OOF ROC-AUC: {roc_auc_score(y, oof_ensemble):.4f}")
print(f"OOF F1-Score: {f1_score(y, (oof_ensemble >= best_thresh).astype(int)):.4f}")

# 6. Corrected Submission Generation
sub = ss.copy()

# Ensure we map our binary predictions and raw probabilities to the correct columns
sub['TargetF1'] = (test_ensemble >= best_thresh).astype(int)
sub['TargetRAUC'] = test_ensemble  # Usually expects raw probabilities for AUC metrics

# Save with exact sample submission structure
sub.to_csv('submission_fixed.csv', index=False)
print("Saved 'submission_fixed.csv' with correct column names!")

Training models across 5 folds...
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[11]	valid_0's auc: 0.828869	valid_0's binary_logloss: 0.587669
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[42]	valid_0's auc: 0.842232	valid_0's binary_logloss: 0.508151
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[114]	valid_0's auc: 0.830879	valid_0's binary_logloss: 0.490346
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[50]	valid_0's auc: 0.799022	valid_0's binary_logloss: 0.530371
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[92]	valid_0's auc: 0.807635	valid_0's binary_logloss: 0.515617

--- Optimized Results ---
Best Threshold: 0.37
OOF Competition Score: 0.8134

--- Results ---
Optimal Threshold: 0.37
OOF Competition Score: 0.8134
OOF ROC-AUC

In [ ]:
import pandas as pd
import numpy as np
import warnings
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 1. Load Data & Merge
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. Advanced Feature Engineering
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['year']  = df['deathdate'].dt.year
    df['month'] = df['deathdate'].dt.month
    df['day']   = df['deathdate'].dt.day
    df['dayofweek'] = df['deathdate'].dt.dayofweek

    # Season mapping
    df['season'] = df['month'].map({
        12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
        6:2, 7:2, 8:2, 9:3, 10:3, 11:3
    })

    # Age features & non-linear transformations
    df['log_age']      = np.log1p(df['age'])
    df['age_squared']  = df['age'] ** 2
    df['age_cubed']    = df['age'] ** 3
    df['is_infant']    = (df['age'] == 0).astype(int)
    df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
    df['is_child']     = (df['age'].between(1, 5)).astype(int)
    df['is_elderly']   = (df['age'] >= 60).astype(int)
    df['age_group']    = pd.cut(df['age'], bins=[-1,0,1,2,5,10,15,30,60,200],
                                labels=[0,1,2,3,4,5,6,7,8]).astype(int)

    # Temperature and climate stress features
    df['temp_range']       = df['max_temperature'] - df['min_temperature']
    df['year_avg_temp']    = df.groupby('year')['max_temperature'].transform('mean')
    df['temp_anomaly']     = df['max_temperature'] - df['year_avg_temp']
    df['temp_ndvi_ratio']  = df['max_temperature'] / (df['ndvi_30d'] + 1e-5)
    df['rain_temp_product']= df['precipitation'] * df['tavg_30d']

    # Historical year sensitivity mapping
    year_sensitivity = {
        2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
        2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
        2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
    }
    df['year_sensitivity'] = df['year'].map(year_sensitivity)
    df['year_sens_age']    = df['year_sensitivity'] * df['age']

    # Categorical encoding
    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp']
    return df.drop(drop_cols, axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.fillna(-999)

# 3. Stratified K-Fold Training with Tuned Hyperparameters
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))

print("Training tuned models...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Finetuned LightGBM
    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.015, max_depth=7, num_leaves=63,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.6,
        subsample=0.85, reg_alpha=0.1, reg_lambda=1.0, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    # Finetuned XGBoost
    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.015, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.6,
        subsample=0.85, gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    # Finetuned CatBoost
    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.02, depth=6,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=3.0,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    # OOF & Test Predictions
    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]

    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

# 4. Optimized Ensemble Blend Weights
oof_ensemble = (0.45 * oof_lgbm) + (0.30 * oof_xgb) + (0.25 * oof_cat)
test_ensemble = (0.45 * test_lgbm) + (0.30 * test_xgb) + (0.25 * test_cat)

# 5. Fine-grained Threshold Optimization
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.005):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Upgraded Ensemble Results ---")
print(f"Best Threshold: {best_thresh:.3f}")
print(f"OOF Competition Score: {best_score:.5f}")
print(f"OOF ROC-AUC: {roc_auc_score(y, oof_ensemble):.5f}")
print(f"OOF F1-Score: {f1_score(y, (oof_ensemble >= best_thresh).astype(int)):.5f}")

# 6. Save Submission File
sub = ss.copy()
sub['TargetF1'] = (test_ensemble >= best_thresh).astype(int)
sub['TargetRAUC'] = test_ensemble
sub.to_csv('submission_087_push.csv', index=False)
print("\nSaved 'submission_087_push.csv' successfully!")

Training tuned models...
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[18]	valid_0's auc: 0.828542	valid_0's binary_logloss: 0.578077
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[27]	valid_0's auc: 0.84099	valid_0's binary_logloss: 0.55354
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[2]	valid_0's auc: 0.841654	valid_0's binary_logloss: 0.63681
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[6]	valid_0's auc: 0.797744	valid_0's binary_logloss: 0.623106
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[116]	valid_0's auc: 0.804257	valid_0's binary_logloss: 0.515112

--- Upgraded Ensemble Results ---
Best Threshold: 0.460
OOF Competition Score: 0.81605
OOF ROC-AUC: 0.81740
OOF F1-Score: 0.81515

Saved 'submission_087_push.csv' successful

In [ ]:
import pandas as pd
import numpy as np
import warnings
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 1. Load Data & Merge
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')


train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. Robust & Efficient Feature Engineering
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['year']  = df['deathdate'].dt.year
    df['month'] = df['deathdate'].dt.month
    df['day']   = df['deathdate'].dt.day
    df['dayofweek'] = df['deathdate'].dt.dayofweek

    # Season mapping
    df['season'] = df['month'].map({
        12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
        6:2, 7:2, 8:2, 9:3, 10:3, 11:3
    })

    # Non-linear Age & Vulnerability features
    df['log_age']      = np.log1p(df['age'])
    df['age_squared']  = df['age'] ** 2
    df['age_cubed']    = df['age'] ** 3
    df['is_infant']    = (df['age'] == 0).astype(int)
    df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
    df['is_child']     = (df['age'].between(1, 5)).astype(int)
    df['is_elderly']   = (df['age'] >= 60).astype(int)
    df['age_group']    = pd.cut(df['age'], bins=[-1,0,1,2,5,10,15,30,60,200],
                                labels=[0,1,2,3,4,5,6,7,8]).astype(int)

    # Climate Stress & Interaction Features
    df['temp_range']       = df['max_temperature'] - df['min_temperature']
    df['year_avg_temp']    = df.groupby('year')['max_temperature'].transform('mean')
    df['temp_anomaly']     = df['max_temperature'] - df['year_avg_temp']
    df['temp_ndvi_ratio']  = df['max_temperature'] / (df['ndvi_30d'] + 1e-5)
    df['rain_temp_product']= df['precipitation'] * df['tavg_30d']

    # Historical year sensitivity mapping
    year_sensitivity = {
        2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
        2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
        2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
    }
    df['year_sensitivity'] = df['year'].map(year_sensitivity)
    df['year_sens_age']    = df['year_sensitivity'] * df['age']

    # Categorical encoding
    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp']
    return df.drop(drop_cols, axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.fillna(-999)

# 3. Stratified K-Fold Cross-Validation Setup
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

# Out-of-fold and test storage matrices for base models
oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))

print("Training base estimators with early stopping...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # LightGBM (Regularized to prevent overfitting)
    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.015, max_depth=6, num_leaves=31,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.65,
        subsample=0.8, reg_alpha=0.2, reg_lambda=1.5, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    # XGBoost (Controlled depth & subsampling)
    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.015, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.65,
        subsample=0.8, gamma=0.2, reg_alpha=0.2, reg_lambda=1.5,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    # CatBoost (Balanced L2 regularization)
    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.02, depth=6,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=5.0,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    # Collect OOF predictions
    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]

    # Collect Test predictions
    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

# 4. Stacking Meta-Classifier (Logistic Regression)
# This learns the optimal dynamic combination of the 3 base models without overfitting.
print("Training Stacking Meta-Classifier...")
X_train_stack = np.column_stack((oof_lgbm, oof_xgb, oof_cat))
X_test_stack  = np.column_stack((test_lgbm, test_xgb, test_cat))

meta_oof = np.zeros(len(X))
meta_test = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(folds.split(X_train_stack, y)):
    X_tr_m, X_val_m = X_train_stack[train_idx], X_train_stack[val_idx]
    y_tr_m, y_val_m = y.iloc[train_idx], y.iloc[val_idx]

    meta_model = LogisticRegression(C=1.0, random_state=42)
    meta_model.fit(X_tr_m, y_tr_m)

    meta_oof[val_idx] = meta_model.predict_proba(X_val_m)[:, 1]
    meta_test += meta_model.predict_proba(X_test_stack)[:, 1] / folds.n_splits

# 5. Threshold Optimization on Meta-OOF
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.005):
    f1 = f1_score(y, (meta_oof >= thresh).astype(int))
    auc = roc_auc_score(y, meta_oof)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Robust Stacked Model Results ---")
print(f"Optimal Decision Threshold: {best_thresh:.3f}")
print(f"Robust OOF Competition Score: {best_score:.5f}")
print(f"OOF ROC-AUC: {roc_auc_score(y, meta_oof):.5f}")
print(f"OOF F1-Score: {f1_score(y, (meta_oof >= best_thresh).astype(int)):.5f}")

# 6. Generate Final Submission File
sub = ss.copy()
sub['TargetF1'] = (meta_test >= best_thresh).astype(int)
sub['TargetRAUC'] = meta_test
sub.to_csv('submission_stacked_efficient.csv', index=False)
print("\nSaved 'submission_stacked_efficient.csv' successfully! Ready to win!")

Training base estimators with early stopping...
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[15]	valid_0's auc: 0.831147	valid_0's binary_logloss: 0.586957
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[8]	valid_0's auc: 0.838345	valid_0's binary_logloss: 0.610029
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[36]	valid_0's auc: 0.835458	valid_0's binary_logloss: 0.541007
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[29]	valid_0's auc: 0.795855	valid_0's binary_logloss: 0.56786
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[153]	valid_0's auc: 0.810336	valid_0's binary_logloss: 0.515144
Training Stacking Meta-Classifier...

--- Robust Stacked Model Results ---
Optimal Decision Threshold: 0.370
Robust OOF Competition Score: 0.82110
O

In [ ]:
import pandas as pd
import numpy as np
import warnings
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 2. Load Data explicitly from the /content/ directory path
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

# Merge climate features using the 'ID' column
train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 3. Robust Feature Engineering Pipeline
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['year']  = df['deathdate'].dt.year
    df['month'] = df['deathdate'].dt.month
    df['day']   = df['deathdate'].dt.day
    df['dayofweek'] = df['deathdate'].dt.dayofweek

    # Season mapping
    df['season'] = df['month'].map({
        12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
        6:2, 7:2, 8:2, 9:3, 10:3, 11:3
    })

    # Non-linear Age & Vulnerability features
    df['log_age']      = np.log1p(df['age'])
    df['age_squared']  = df['age'] ** 2
    df['age_cubed']    = df['age'] ** 3
    df['is_infant']    = (df['age'] == 0).astype(int)
    df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
    df['is_child']     = (df['age'].between(1, 5)).astype(int)
    df['is_elderly']   = (df['age'] >= 60).astype(int)
    df['age_group']    = pd.cut(df['age'], bins=[-1,0,1,2,5,10,15,30,60,200],
                                labels=[0,1,2,3,4,5,6,7,8]).astype(int)

    # Climate Stress & Interaction Features
    df['temp_range']       = df['max_temperature'] - df['min_temperature']
    df['year_avg_temp']    = df.groupby('year')['max_temperature'].transform('mean')
    df['temp_anomaly']     = df['max_temperature'] - df['year_avg_temp']
    df['temp_ndvi_ratio']  = df['max_temperature'] / (df['ndvi_30d'] + 1e-5)
    df['rain_temp_product']= df['precipitation'] * df['tavg_30d']

    # Historical year sensitivity mapping
    year_sensitivity = {
        2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
        2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
        2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
    }
    df['year_sensitivity'] = df['year'].map(year_sensitivity)
    df['year_sens_age']    = df['year_sensitivity'] * df['age']

    # Categorical encoding
    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp']
    return df.drop(drop_cols, axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.fillna(-999)

# 4. Stratified K-Fold Training with Balanced Learning Rates
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))

print("Training base models...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # LightGBM
    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.02, max_depth=6, num_leaves=31,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.7,
        subsample=0.85, reg_alpha=0.1, reg_lambda=1.0, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    # XGBoost
    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.02, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.7,
        subsample=0.85, gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    # CatBoost
    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.025, depth=6,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=3.0,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    # OOF Predictions
    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]

    # Test Predictions
    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

# 5. Optimized Weighted Ensemble Blend (Preserves probability calibration)
oof_ensemble = (0.45 * oof_lgbm) + (0.35 * oof_xgb) + (0.20 * oof_cat)
test_ensemble = (0.45 * test_lgbm) + (0.35 * test_xgb) + (0.20 * test_cat)

# 6. Fine-grained Threshold Optimization
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.001):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Final Optimized Ensemble Results ---")
print(f"Best Decision Threshold: {best_thresh:.3f}")
print(f"OOF Competition Score: {best_score:.5f}")
print(f"OOF ROC-AUC: {roc_auc_score(y, oof_ensemble):.5f}")
print(f"OOF F1-Score: {f1_score(y, (oof_ensemble >= best_thresh).astype(int)):.5f}")

# 7. Save Submission File & Trigger Download
sub = ss.copy()
sub['TargetF1'] = (test_ensemble >= best_thresh).astype(int)
sub['TargetRAUC'] = test_ensemble
sub_filename = '/content/submission_optimized_final.csv'
sub.to_csv(sub_filename, index=False)
print(f"\nSaved '{sub_filename}' successfully!")

# Automatically trigger file download in Colab
from google.colab import files
files.download(sub_filename)

Training base models...
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[20]	valid_0's auc: 0.823043	valid_0's binary_logloss: 0.55742
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[76]	valid_0's auc: 0.836842	valid_0's binary_logloss: 0.487231
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[3]	valid_0's auc: 0.836558	valid_0's binary_logloss: 0.627421
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[2]	valid_0's auc: 0.801678	valid_0's binary_logloss: 0.63515
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[92]	valid_0's auc: 0.807257	valid_0's binary_logloss: 0.514774

--- Final Optimized Ensemble Results ---
Best Decision Threshold: 0.381
OOF Competition Score: 0.80846
OOF ROC-AUC: 0.81012
OOF F1-Score: 0.80736

Saved '/content/submission_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import warnings
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 2. Load Data explicitly from the /content/ directory path
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

# Merge climate features using the 'ID' column
train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 3. High-Performance Geospatial & Climate Feature Engineering
def engineer_features(train_df, test_df):
    # Combine for global spatial clustering
    combined = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    # K-Means Spatial Clusters based on Latitude & Longitude
    kmeans = KMeans(n_clusters=10, random_state=42).fit(combined[['latitude', 'longitude']])
    combined['geo_cluster'] = kmeans.predict(combined[['latitude', 'longitude']])

    # Split back
    train_feat = combined.iloc[:len(train_df)].copy()
    test_feat  = combined.iloc[len(train_df):].copy()

    def process(df):
        df = df.copy()
        df['deathdate'] = pd.to_datetime(df['deathdate'])
        df['year']  = df['deathdate'].dt.year
        df['month'] = df['deathdate'].dt.month
        df['day']   = df['deathdate'].dt.day
        df['dayofweek'] = df['deathdate'].dt.dayofweek

        # Season mapping
        df['season'] = df['month'].map({
            12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
            6:2, 7:2, 8:2, 9:3, 10:3, 11:3
        })

        # Non-linear Age & Vulnerability features
        df['log_age']      = np.log1p(df['age'])
        df['age_squared']  = df['age'] ** 2
        df['age_cubed']    = df['age'] ** 3
        df['is_infant']    = (df['age'] == 0).astype(int)
        df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
        df['is_child']     = (df['age'].between(1, 5)).astype(int)
        df['is_elderly']   = (df['age'] >= 60).astype(int)

        # Climate & Geospatial Interactions
        df['temp_range']       = df['max_temperature'] - df['min_temperature']
        df['year_avg_temp']    = df.groupby('year')['max_temperature'].transform('mean')
        df['temp_anomaly']     = df['max_temperature'] - df['year_avg_temp']
        df['temp_ndvi_ratio']  = df['max_temperature'] / (df['ndvi_30d'] + 1e-5)
        df['rain_temp_product']= df['precipitation'] * df['tavg_30d']
        df['lat_lon_product']  = df['latitude'] * df['longitude']

        # Acute Weather Stress
        df['heat_stress_index']= df['hot_days_30d'] * df['max_temperature']
        df['drought_index']    = df['rain_sum_30d'] / (df['ndvi_30d'] + 1e-5)
        df['age_heat_interaction'] = df['age'] * df['hot_days_30d']

        # Historical year sensitivity mapping
        year_sensitivity = {
            2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
            2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
            2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
        }
        df['year_sensitivity'] = df['year'].map(year_sensitivity)
        df['year_sens_age']    = df['year_sensitivity'] * df['age']

        # Categorical encoding
        df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
        df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

        drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp']
        return df.drop(drop_cols, axis=1, errors='ignore')

    return process(train_feat), process(test_feat)

train_df, test_df = engineer_features(train, test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.drop('is_climate_sensitive', axis=1, errors='ignore').fillna(-999)

# 4. Stratified K-Fold Training
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))

print("Training base models with geospatial features...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # LightGBM
    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.018, max_depth=6, num_leaves=35,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.75,
        subsample=0.85, reg_alpha=0.05, reg_lambda=0.8, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    # XGBoost
    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.018, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.75,
        subsample=0.85, gamma=0.05, reg_alpha=0.05, reg_lambda=0.8,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    # CatBoost
    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.02, depth=6,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=2.5,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    # OOF Predictions
    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]

    # Test Predictions
    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

# 5. Optimized Weighted Ensemble Blend
oof_ensemble = (0.45 * oof_lgbm) + (0.35 * oof_xgb) + (0.20 * oof_cat)
test_ensemble = (0.45 * test_lgbm) + (0.35 * test_xgb) + (0.20 * test_cat)

# 6. Fine-grained Threshold Optimization
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.001):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Geospatial Enhanced Results ---")
print(f"Best Decision Threshold: {best_thresh:.3f}")
print(f"OOF Competition Score: {best_score:.5f}")
print(f"OOF ROC-AUC: {roc_auc_score(y, oof_ensemble):.5f}")
print(f"OOF F1-Score: {f1_score(y, (oof_ensemble >= best_thresh).astype(int)):.5f}")

# 7. Save Submission File & Trigger Download
sub = ss.copy()
sub['TargetF1'] = (test_ensemble >= best_thresh).astype(int)
sub['TargetRAUC'] = test_ensemble
sub_filename = '/content/submission_geospatial_boost.csv'
sub.to_csv(sub_filename, index=False)
print(f"\nSaved '{sub_filename}' successfully!")

from google.colab import files
files.download(sub_filename)

Training base models with geospatial features...
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[16]	valid_0's auc: 0.827744	valid_0's binary_logloss: 0.572865
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[28]	valid_0's auc: 0.839436	valid_0's binary_logloss: 0.538252
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[16]	valid_0's auc: 0.831463	valid_0's binary_logloss: 0.573868
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[32]	valid_0's auc: 0.794916	valid_0's binary_logloss: 0.553474
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[91]	valid_0's auc: 0.805657	valid_0's binary_logloss: 0.516118

--- Geospatial Enhanced Results ---
Best Decision Threshold: 0.406
OOF Competition Score: 0.81465
OOF ROC-AUC: 0.81332
OOF F1-Score: 0.81553

Sav

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import warnings
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')


# 2. Load Data explicitly from the /content/ directory path
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

# Merge climate features using the 'ID' column
train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 3. Advanced Geospatial & Climate Feature Engineering
def engineer_features(train_df, test_df):
    combined = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    # K-Means Spatial Clusters
    kmeans = KMeans(n_clusters=12, random_state=42).fit(combined[['latitude', 'longitude']])
    combined['geo_cluster'] = kmeans.predict(combined[['latitude', 'longitude']])

    train_feat = combined.iloc[:len(train_df)].copy()
    test_feat  = combined.iloc[len(train_df):].copy()

    def process(df):
        df = df.copy()
        df['deathdate'] = pd.to_datetime(df['deathdate'])
        df['year']  = df['deathdate'].dt.year
        df['month'] = df['deathdate'].dt.month
        df['day']   = df['deathdate'].dt.day
        df['dayofweek'] = df['deathdate'].dt.dayofweek

        # Season mapping
        df['season'] = df['month'].map({
            12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
            6:2, 7:2, 8:2, 9:3, 10:3, 11:3
        })

        # Non-linear Age & Vulnerability features
        df['log_age']      = np.log1p(df['age'])
        df['age_squared']  = df['age'] ** 2
        df['age_cubed']    = df['age'] ** 3
        df['is_infant']    = (df['age'] == 0).astype(int)
        df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
        df['is_child']     = (df['age'].between(1, 5)).astype(int)
        df['is_elderly']   = (df['age'] >= 60).astype(int)

        # Climate & Geospatial Interactions
        df['temp_range']       = df['max_temperature'] - df['min_temperature']
        df['year_avg_temp']    = df.groupby('year')['max_temperature'].transform('mean')
        df['temp_anomaly']     = df['max_temperature'] - df['year_avg_temp']
        df['temp_ndvi_ratio']  = df['max_temperature'] / (df['ndvi_30d'] + 1e-5)
        df['rain_temp_product']= df['precipitation'] * df['tavg_30d']
        df['lat_lon_product']  = df['latitude'] * df['longitude']

        # Acute Weather Stress
        df['heat_stress_index']= df['hot_days_30d'] * df['max_temperature']
        df['drought_index']    = df['rain_sum_30d'] / (df['ndvi_30d'] + 1e-5)
        df['age_heat_interaction'] = df['age'] * df['hot_days_30d']

        # Historical year sensitivity mapping
        year_sensitivity = {
            2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
            2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
            2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
        }
        df['year_sensitivity'] = df['year'].map(year_sensitivity)
        df['year_sens_age']    = df['year_sensitivity'] * df['age']

        # Categorical encoding
        df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
        df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

        drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp']
        return df.drop(drop_cols, axis=1, errors='ignore')

    return process(train_feat), process(test_feat)

train_df, test_df = engineer_features(train, test)

# 4. Out-of-Fold Target Encoding for Spatial Clusters to prevent leakage
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_df['cluster_target_enc'] = np.nan
test_df['cluster_target_enc']  = np.nan

for train_idx, val_idx in folds.split(train_df, train_df['is_climate_sensitive']):
    tr_fold = train_df.iloc[train_idx]
    val_fold = train_df.iloc[val_idx]

    # Calculate mean target per cluster on training fold
    encoding_map = tr_fold.groupby('geo_cluster')['is_climate_sensitive'].mean()
    train_df.iloc[val_idx, train_df.columns.get_loc('cluster_target_enc')] = val_fold['geo_cluster'].map(encoding_map)

# Fill overall mean for train training folds and test set
global_mean = train_df['is_climate_sensitive'].mean()
train_df['cluster_target_enc'].fillna(global_mean, inplace=True)

cluster_full_map = train_df.groupby('geo_cluster')['is_climate_sensitive'].mean()
test_df['cluster_target_enc'] = test_df['geo_cluster'].map(cluster_full_map).fillna(global_mean)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.drop('is_climate_sensitive', axis=1, errors='ignore').fillna(-999)

neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))

print("Training models with Target-Encoded Geospatial features...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # LightGBM
    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.015, max_depth=6, num_leaves=35,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.75,
        subsample=0.85, reg_alpha=0.05, reg_lambda=0.8, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    # XGBoost
    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.015, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.75,
        subsample=0.85, gamma=0.05, reg_alpha=0.05, reg_lambda=0.8,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    # CatBoost
    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.02, depth=6,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=2.5,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    # OOF Predictions
    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]

    # Test Predictions
    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

# 5. Optimized Weighted Ensemble Blend
oof_ensemble = (0.45 * oof_lgbm) + (0.35 * oof_xgb) + (0.20 * oof_cat)
test_ensemble = (0.45 * test_lgbm) + (0.35 * test_xgb) + (0.20 * test_cat)

# 6. Fine-grained Threshold Optimization
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.001):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Target-Encoded Results ---")
print(f"Best Decision Threshold: {best_thresh:.3f}")
print(f"OOF Competition Score: {best_score:.5f}")
print(f"OOF ROC-AUC: {roc_auc_score(y, oof_ensemble):.5f}")
print(f"OOF F1-Score: {f1_score(y, (oof_ensemble >= best_thresh).astype(int)):.5f}")

# 7. Save Submission File & Trigger Download
sub = ss.copy()
sub['TargetF1'] = (test_ensemble >= best_thresh).astype(int)
sub['TargetRAUC'] = test_ensemble
sub_filename = '/content/submission_target_encoded.csv'
sub.to_csv(sub_filename, index=False)
print(f"\nSaved '{sub_filename}' successfully!")

from google.colab import files
files.download(sub_filename)

Training models with Target-Encoded Geospatial features...
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[15]	valid_0's auc: 0.82597	valid_0's binary_logloss: 0.585287
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[36]	valid_0's auc: 0.845155	valid_0's binary_logloss: 0.530982
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[154]	valid_0's auc: 0.830212	valid_0's binary_logloss: 0.490501
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[72]	valid_0's auc: 0.794882	valid_0's binary_logloss: 0.529607
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[115]	valid_0's auc: 0.808524	valid_0's binary_logloss: 0.513313

--- Target-Encoded Results ---
Best Decision Threshold: 0.389
OOF Competition Score: 0.81690
OOF ROC-AUC: 0.81671
OOF F1-Score: 0.8170

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import warnings
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 1. Load Data explicitly from the /content/ directory path
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. Advanced Environmental Shock & Context Engineering
def engineer_features(train_df, test_df):
    combined = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    # K-Means Spatial Clusters
    kmeans = KMeans(n_clusters=12, random_state=42).fit(combined[['latitude', 'longitude']])
    combined['geo_cluster'] = kmeans.predict(combined[['latitude', 'longitude']])

    train_feat = combined.iloc[:len(train_df)].copy()
    test_feat  = combined.iloc[len(train_df):].copy()

    def process(df):
        df = df.copy()
        df['deathdate'] = pd.to_datetime(df['deathdate'])
        df['year']  = df['deathdate'].dt.year
        df['month'] = df['deathdate'].dt.month
        df['day']   = df['deathdate'].dt.day
        df['dayofweek'] = df['deathdate'].dt.dayofweek

        # Season mapping
        df['season'] = df['month'].map({
            12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
            6:2, 7:2, 8:2, 9:3, 10:3, 11:3
        })

        # Non-linear Age Features
        df['log_age']      = np.log1p(df['age'])
        df['age_squared']  = df['age'] ** 2
        df['is_infant']    = (df['age'] == 0).astype(int)
        df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
        df['is_elderly']   = (df['age'] >= 60).astype(int)

        # NEW ANGLE 1: Climate Shocks (Short-term vs Long-term Deltas)
        df['temp_shock_7_90'] = df['tavg_7d'] - df['tavg_90d']
        df['rain_shock_7_90'] = df['rain_sum_7d'] - df['rain_sum_90d']
        df['temp_range']      = df['max_temperature'] - df['min_temperature']

        # NEW ANGLE 2: Biological Tipping Points (High Risk Crossings)
        df['elderly_heat_risk'] = ((df['age'] >= 60) & (df['hot_days_30d'] > df['hot_days_30d'].median())).astype(int)
        df['infant_rain_risk']  = ((df['age'] <= 2) & (df['rain_sum_30d'] > df['rain_sum_30d'].median())).astype(int)

        # Geospatial & Weather Interactions
        df['heat_stress_index'] = df['hot_days_30d'] * df['max_temperature']
        df['drought_index']     = df['rain_sum_30d'] / (df['ndvi_30d'] + 1e-5)
        df['lat_lon_product']   = df['latitude'] * df['longitude']

        # Historical year sensitivity mapping
        year_sensitivity = {
            2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
            2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
            2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
        }
        df['year_sensitivity'] = df['year'].map(year_sensitivity)
        df['year_sens_age']    = df['year_sensitivity'] * df['age']

        # Categorical encoding
        df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
        df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

        drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra']
        return df.drop(drop_cols, axis=1, errors='ignore')

    return process(train_feat), process(test_feat)

train_df, test_df = engineer_features(train, test)

# NEW ANGLE 3: Relative Zone Deviations (Contextual Aggregations)
zone_temp_means = train_df.groupby('zone')['max_temperature'].mean().to_dict()
train_df['zone_temp_deviation'] = train_df['zone'].map(zone_temp_means) - train_df['max_temperature']
test_df['zone_temp_deviation']  = test_df['zone'].map(zone_temp_means) - test_df['max_temperature']

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.drop('is_climate_sensitive', axis=1, errors='ignore').fillna(-999)

# 3. Stratified K-Fold Training
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))

print("Training models with Environmental Shock features...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # LightGBM
    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.015, max_depth=6, num_leaves=35,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.75,
        subsample=0.85, reg_alpha=0.05, reg_lambda=0.8, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    # XGBoost
    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.015, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.75,
        subsample=0.85, gamma=0.05, reg_alpha=0.05, reg_lambda=0.8,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    # CatBoost
    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.02, depth=6,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=2.5,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    # OOF Predictions
    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]

    # Test Predictions
    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

# 4. Optimized Weighted Ensemble Blend
oof_ensemble = (0.45 * oof_lgbm) + (0.35 * oof_xgb) + (0.20 * oof_cat)
test_ensemble = (0.45 * test_lgbm) + (0.35 * test_xgb) + (0.20 * test_cat)

# 5. Fine-grained Threshold Optimization
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.001):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Shock-Engineered Results ---")
print(f"Best Decision Threshold: {best_thresh:.3f}")
print(f"OOF Competition Score: {best_score:.5f}")
print(f"OOF ROC-AUC: {roc_auc_score(y, oof_ensemble):.5f}")
print(f"OOF F1-Score: {f1_score(y, (oof_ensemble >= best_thresh).astype(int)):.5f}")

# 6. Save Submission File & Trigger Download
sub = ss.copy()
sub['TargetF1'] = (test_ensemble >= best_thresh).astype(int)
sub['TargetRAUC'] = test_ensemble
sub_filename = '/content/submission_climate_shocks.csv'
sub.to_csv(sub_filename, index=False)
print(f"\nSaved '{sub_filename}' successfully!")

from google.colab import files
files.download(sub_filename)

Training models with Environmental Shock features...
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[22]	valid_0's auc: 0.83316	valid_0's binary_logloss: 0.566105
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[38]	valid_0's auc: 0.839008	valid_0's binary_logloss: 0.531538
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[47]	valid_0's auc: 0.832885	valid_0's binary_logloss: 0.523154
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[9]	valid_0's auc: 0.794482	valid_0's binary_logloss: 0.612446
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[117]	valid_0's auc: 0.80439	valid_0's binary_logloss: 0.515635

--- Shock-Engineered Results ---
Best Decision Threshold: 0.453
OOF Competition Score: 0.81403
OOF ROC-AUC: 0.81612
OOF F1-Score: 0.81263

Save

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import warnings
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 1. Load Data
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. Clean, Core Feature Engineering (No bloat)
def engineer_features(train_df, test_df):
    combined = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    kmeans = KMeans(n_clusters=10, random_state=42).fit(combined[['latitude', 'longitude']])
    combined['geo_cluster'] = kmeans.predict(combined[['latitude', 'longitude']])

    train_feat = combined.iloc[:len(train_df)].copy()
    test_feat  = combined.iloc[len(train_df):].copy()

    def process(df):
        df = df.copy()
        df['deathdate'] = pd.to_datetime(df['deathdate'])
        df['year']  = df['deathdate'].dt.year
        df['month'] = df['deathdate'].dt.month

        # Core interactions that matter
        df['log_age']     = np.log1p(df['age'])
        df['is_infant']   = (df['age'] <= 2).astype(int)
        df['is_elderly']  = (df['age'] >= 60).astype(int)
        df['temp_range']  = df['max_temperature'] - df['min_temperature']
        df['heat_stress'] = df['hot_days_30d'] * df['max_temperature']
        df['drought_idx'] = df['rain_sum_30d'] / (df['ndvi_30d'] + 1e-5)

        df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
        df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

        drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra']
        return df.drop(drop_cols, axis=1, errors='ignore')

    return process(train_feat), process(test_feat)

train_df, test_df = engineer_features(train, test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.drop('is_climate_sensitive', axis=1, errors='ignore').fillna(-999)

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))

print("Training with Heavy Regularization to beat the noise wall...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Stricter regularization to prevent overfitting
    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=4, num_leaves=15,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.6,
        subsample=0.75, reg_alpha=0.5, reg_lambda=2.0, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=4,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.6,
        subsample=0.75, gamma=0.2, reg_alpha=0.5, reg_lambda=2.0,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.015, depth=4,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=5.0,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]

    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

oof_ensemble = (0.45 * oof_lgbm) + (0.35 * oof_xgb) + (0.20 * oof_cat)
test_ensemble = (0.45 * test_lgbm) + (0.35 * test_xgb) + (0.20 * test_cat)

best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.001):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Regularized Results ---")
print(f"Best Decision Threshold: {best_thresh:.3f}")
print(f"OOF Competition Score: {best_score:.5f}")

sub = ss.copy()
sub['TargetF1'] = (test_ensemble >= best_thresh).astype(int)
sub['TargetRAUC'] = test_ensemble
sub.to_csv('/content/submission_regularized.csv', index=False)

from google.colab import files
files.download('/content/submission_regularized.csv')

Training with Heavy Regularization to beat the noise wall...
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[46]	valid_0's auc: 0.838049	valid_0's binary_logloss: 0.55457
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[115]	valid_0's auc: 0.834497	valid_0's binary_logloss: 0.508158
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[264]	valid_0's auc: 0.825333	valid_0's binary_logloss: 0.504289
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[53]	valid_0's auc: 0.799205	valid_0's binary_logloss: 0.564151
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[202]	valid_0's auc: 0.807057	valid_0's binary_logloss: 0.519019

--- Regularized Results ---
Best Decision Threshold: 0.400
OOF Competition Score: 0.81774


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import warnings
from scipy.optimize import minimize
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 1. Load Data
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. Clean Feature Engineering
def engineer_features(train_df, test_df):
    combined = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    kmeans = KMeans(n_clusters=12, random_state=42).fit(combined[['latitude', 'longitude']])
    combined['geo_cluster'] = kmeans.predict(combined[['latitude', 'longitude']])

    train_feat = combined.iloc[:len(train_df)].copy()
    test_feat  = combined.iloc[len(train_df):].copy()

    def process(df):
        df = df.copy()
        df['deathdate'] = pd.to_datetime(df['deathdate'])
        df['year']  = df['deathdate'].dt.year
        df['month'] = df['deathdate'].dt.month

        # Core non-linear & interaction features
        df['log_age']       = np.log1p(df['age'])
        df['is_infant']     = (df['age'] <= 2).astype(int)
        df['is_elderly']    = (df['age'] >= 60).astype(int)
        df['temp_range']    = df['max_temperature'] - df['min_temperature']
        df['heat_stress']   = df['hot_days_30d'] * df['max_temperature']
        df['drought_idx']   = df['rain_sum_30d'] / (df['ndvi_30d'] + 1e-5)
        df['temp_ndvi']     = df['max_temperature'] / (df['ndvi_30d'] + 1e-5)

        year_sensitivity = {
            2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
            2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
            2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
        }
        df['year_sensitivity'] = df['year'].map(year_sensitivity)

        df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
        df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

        drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra']
        return df.drop(drop_cols, axis=1, errors='ignore')

    return process(train_feat), process(test_feat)

train_df, test_df = engineer_features(train, test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.drop('is_climate_sensitive', axis=1, errors='ignore').fillna(-999)

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))

print("Training Diverse Base Models...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Model 1: LightGBM (Deep & agile)
    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=6, num_leaves=40,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.7,
        subsample=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    # Model 2: XGBoost (Conservative split)
    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.6,
        subsample=0.8, gamma=0.1, reg_alpha=0.2, reg_lambda=1.5,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    # Model 3: CatBoost (Symmetric trees)
    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.015, depth=5,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=3.0,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]

    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

# 3. Numerical Optimization to Find Best Ensemble Weights
def objective_function(weights):
    w1, w2, w3 = weights
    blend = (w1 * oof_lgbm) + (w2 * oof_xgb) + (w3 * oof_cat)
    auc = roc_auc_score(y, blend)
    return -auc  # Maximize AUC

res = minimize(objective_function, [0.33, 0.33, 0.33], bounds=[(0,1), (0,1), (0,1)], constraints={'type': 'eq', 'fun': lambda w: 1 - sum(w)})
w_lgbm, w_xgb, w_cat = res.x

print(f"\nOptimal Stacked Weights Found -> LGBM: {w_lgbm:.3f}, XGB: {w_xgb:.3f}, Cat: {w_cat:.3f}")

oof_ensemble = (w_lgbm * oof_lgbm) + (w_xgb * oof_xgb) + (w_cat * oof_cat)
test_ensemble = (w_lgbm * test_lgbm) + (w_xgb * test_xgb) + (w_cat * test_cat)

# 4. Threshold Optimization for F1/AUC Score
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.001):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Optimized Stacking Results ---")
print(f"Best Decision Threshold: {best_thresh:.3f}")
print(f"OOF Competition Score: {best_score:.5f}")
print(f"OOF ROC-AUC: {roc_auc_score(y, oof_ensemble):.5f}")

# 5. Save Submission
sub = ss.copy()
sub['TargetF1'] = (test_ensemble >= best_thresh).astype(int)
sub['TargetRAUC'] = test_ensemble
sub.to_csv('/content/submission_optimized_stack.csv', index=False)
print("\nSaved '/content/submission_optimized_stack.csv' successfully!")

from google.colab import files
files.download('/content/submission_optimized_stack.csv')

Training Diverse Base Models...
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[3]	valid_0's auc: 0.827422	valid_0's binary_logloss: 0.637627
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[109]	valid_0's auc: 0.844988	valid_0's binary_logloss: 0.495634
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[7]	valid_0's auc: 0.834974	valid_0's binary_logloss: 0.625295
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[11]	valid_0's auc: 0.805935	valid_0's binary_logloss: 0.619389
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[170]	valid_0's auc: 0.806468	valid_0's binary_logloss: 0.515582

Optimal Stacked Weights Found -> LGBM: 0.333, XGB: 0.333, Cat: 0.333

--- Optimized Stacking Results ---
Best Decision Threshold: 0.412
OOF Competition Score: 0.8

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import warnings
from scipy.optimize import minimize
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 1. Load Data
print("Loading datasets...")
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. Feature Engineering Pipeline
def engineer_features(train_df, test_df):
    combined = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    kmeans = KMeans(n_clusters=12, random_state=42).fit(combined[['latitude', 'longitude']])
    combined['geo_cluster'] = kmeans.predict(combined[['latitude', 'longitude']])

    train_feat = combined.iloc[:len(train_df)].copy()
    test_feat  = combined.iloc[len(train_df):].copy()

    def process(df):
        df = df.copy()
        df['deathdate'] = pd.to_datetime(df['deathdate'])
        df['year']  = df['deathdate'].dt.year
        df['month'] = df['deathdate'].dt.month

        # Core non-linear & interaction features
        df['log_age']       = np.log1p(df['age'])
        df['is_infant']     = (df['age'] <= 2).astype(int)
        df['is_elderly']    = (df['age'] >= 60).astype(int)
        df['temp_range']    = df['max_temperature'] - df['min_temperature']
        df['heat_stress']   = df['hot_days_30d'] * df['max_temperature']
        df['drought_idx']   = df['rain_sum_30d'] / (df['ndvi_30d'] + 1e-5)
        df['temp_ndvi']     = df['max_temperature'] / (df['ndvi_30d'] + 1e-5)

        year_sensitivity = {
            2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
            2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
            2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
        }
        df['year_sensitivity'] = df['year'].map(year_sensitivity)

        df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
        df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

        drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra']
        return df.drop(drop_cols, axis=1, errors='ignore')

    return process(train_feat), process(test_feat)

train_df, test_df = engineer_features(train, test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.drop('is_climate_sensitive', axis=1, errors='ignore').fillna(-999)

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

# --- ROUND 1: Train Base Models & Generate Initial Test Blend ---
oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))

print("\n--- Running Round 1 Base Model Training ---")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=6, num_leaves=40,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.7,
        subsample=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.6,
        subsample=0.8, gamma=0.1, reg_alpha=0.2, reg_lambda=1.5,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.015, depth=5,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=3.0,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]

    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

# Optimize weights for Round 1 blend
def objective_function(weights):
    w1, w2, w3 = weights
    blend = (w1 * oof_lgbm) + (w2 * oof_xgb) + (w3 * oof_cat)
    return -roc_auc_score(y, blend)

res = minimize(objective_function, [0.33, 0.33, 0.33], bounds=[(0,1), (0,1), (0,1)], constraints={'type': 'eq', 'fun': lambda w: 1 - sum(w)})
w_lgbm, w_xgb, w_cat = res.x
test_ensemble_r1 = (w_lgbm * test_lgbm) + (w_xgb * test_xgb) + (w_cat * test_cat)


# --- ROUND 2: Pseudo-Labeling & Final Refit ---
print("\n--- Extracting High-Confidence Pseudo-Labels ---")
high_conf_pos = test_ensemble_r1 > 0.88
high_conf_neg = test_ensemble_r1 < 0.12

pseudo_X = X_test[high_conf_pos | high_conf_neg].copy()
pseudo_y = (test_ensemble_r1[high_conf_pos | high_conf_neg] > 0.5).astype(int)
print(f"Injected {len(pseudo_X)} high-confidence rows from test set.")

X_extended = pd.concat([X, pseudo_X], axis=0).reset_index(drop=True)
y_extended = pd.concat([y, pd.Series(pseudo_y)], axis=0).reset_index(drop=True)

# Final Training Loop with Extended Dataset
oof_lgbm_final = np.zeros(len(X))
oof_xgb_final  = np.zeros(len(X))
oof_cat_final  = np.zeros(len(X))

test_lgbm_final = np.zeros(len(X_test))
test_xgb_final  = np.zeros(len(X_test))
test_cat_final  = np.zeros(len(X_test))

print("\n--- Running Final Refit with Pseudo-Labels ---")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)): # Note: validate on original X indices to keep OOF honest
    # Train on extended dataset, validate on original validation fold
    train_mask = ~np.isin(np.arange(len(X_extended)), val_idx + len(X)) # simplified split logic or use standard indices
    # To keep validation strictly on original train rows:
    X_tr = X_extended.drop(val_idx, axis=0)
    y_tr = y_extended.drop(val_idx, axis=0)
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=6, num_leaves=40,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.7,
        subsample=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.6,
        subsample=0.8, gamma=0.1, reg_alpha=0.2, reg_lambda=1.5,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.015, depth=5,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=3.0,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    oof_lgbm_final[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb_final[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat_final[val_idx]  = cat.predict_proba(X_val)[:, 1]

    test_lgbm_final += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb_final  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat_final  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

# Final Optimization Weights
def objective_final(weights):
    w1, w2, w3 = weights
    blend = (w1 * oof_lgbm_final) + (w2 * oof_xgb_final) + (w3 * oof_cat_final)
    return -roc_auc_score(y, blend)

res_final = minimize(objective_final, [w_lgbm, w_xgb, w_cat], bounds=[(0,1), (0,1), (0,1)], constraints={'type': 'eq', 'fun': lambda w: 1 - sum(w)})
wf_lgbm, wf_xgb, wf_cat = res_final.x

oof_ensemble_final = (wf_lgbm * oof_lgbm_final) + (wf_xgb * oof_xgb_final) + (wf_cat * oof_cat_final)
test_ensemble_final = (wf_lgbm * test_lgbm_final) + (wf_xgb * test_xgb_final) + (wf_cat * test_cat_final)

# Threshold Search
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.001):
    f1 = f1_score(y, (oof_ensemble_final >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble_final)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Final Pseudo-Labeled Stacking Results ---")
print(f"Best Decision Threshold: {best_thresh:.3f}")
print(f"Final OOF Competition Score: {best_score:.5f}")

# Save and Download
sub = ss.copy()
sub['TargetF1'] = (test_ensemble_final >= best_thresh).astype(int)
sub['TargetRAUC'] = test_ensemble_final
sub_filename = '/content/submission_final_pseudolabeled.csv'
sub.to_csv(sub_filename, index=False)
print(f"\nSaved '{sub_filename}' successfully!")

from google.colab import files
files.download(sub_filename)

Loading datasets...

--- Running Round 1 Base Model Training ---
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[3]	valid_0's auc: 0.827422	valid_0's binary_logloss: 0.637627
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[109]	valid_0's auc: 0.844988	valid_0's binary_logloss: 0.495634
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[7]	valid_0's auc: 0.834974	valid_0's binary_logloss: 0.625295
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[11]	valid_0's auc: 0.805935	valid_0's binary_logloss: 0.619389
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[170]	valid_0's auc: 0.806468	valid_0's binary_logloss: 0.515582

--- Extracting High-Confidence Pseudo-Labels ---
Injected 0 high-confidence rows from test set.

--- Running Final Refit with Pse

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. Inspect Feature Importances from our LightGBM model to see what's driving the score
import matplotlib.pyplot as plt
import seaborn as sns

feature_importances = pd.DataFrame({
    'feature': X.columns,
    'importance': lgbm.feature_importances_
}).sort_values('importance', ascending=False)

print("--- Top 10 Most Important Features ---")
print(feature_importances.head(10))

# 2. Train with slightly more capacity (Deeper trees) to capture complex patterns
print("\nTraining with higher tree capacity (Deeper leaves)...")
oof_lgbm_deep = np.zeros(len(X))
test_lgbm_deep = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lgbm_deep = LGBMClassifier(
        n_estimators=3000, learning_rate=0.015, max_depth=7, num_leaves=63,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.8,
        subsample=0.85, reg_alpha=0.05, reg_lambda=0.5, random_state=42
    )
    lgbm_deep.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
                  callbacks=[early_stopping(200), log_evaluation(0)])

    oof_lgbm_deep[val_idx] = lgbm_deep.predict_proba(X_val)[:, 1]
    test_lgbm_deep += lgbm_deep.predict_proba(X_test)[:, 1] / folds.n_splits

print(f"Deep LGBM OOF AUC: {roc_auc_score(y, oof_lgbm_deep):.5f}")

--- Top 10 Most Important Features ---
             feature  importance
2                age         725
26              year         318
35  year_sensitivity         273
7           latitude         268
13          ndvi_90d         263
6      precipitation         257
17      rain_sum_90d         251
21          tavg_90d         228
19          tavg_30d         216
31        temp_range         213

Training with higher tree capacity (Deeper leaves)...
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[18]	valid_0's auc: 0.822927	valid_0's binary_logloss: 0.577629
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[57]	valid_0's auc: 0.841937	valid_0's binary_logloss: 0.504747
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[21]	valid_0's auc: 0.824478	valid_0's binary_logloss: 0.570883
Training until validation scores don't improve for 200 round

In [ ]:
# Train high-capacity LightGBM with strict leaf constraints to prevent early-stopping collapse
print("Training High-Capacity LGBM with Controlled Regularization...")
oof_lgbm_controlled = np.zeros(len(X))
test_lgbm_controlled = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lgbm_ctrl = LGBMClassifier(
        n_estimators=3000,
        learning_rate=0.008,          # Slower learning rate
        max_depth=6,
        num_leaves=48,
        min_child_samples=50,         # Stops leaves from splitting on micro-groups (prevents early stopping crash)
        scale_pos_weight=scale_pos_weight_val,
        colsample_bytree=0.7,
        subsample=0.8,
        reg_alpha=0.2,
        reg_lambda=1.5,
        random_state=42
    )
    lgbm_ctrl.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[early_stopping(300), log_evaluation(0)] # Give it more room to breathe before stopping
    )

    oof_lgbm_controlled[val_idx] = lgbm_ctrl.predict_proba(X_val)[:, 1]
    test_lgbm_controlled += lgbm_ctrl.predict_proba(X_test)[:, 1] / folds.n_splits

print(f"Controlled High-Capacity LGBM OOF AUC: {roc_auc_score(y, oof_lgbm_controlled):.5f}")

Training High-Capacity LGBM with Controlled Regularization...
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[45]	valid_0's auc: 0.834762	valid_0's binary_logloss: 0.563122
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[176]	valid_0's auc: 0.840762	valid_0's binary_logloss: 0.491898
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[82]	valid_0's auc: 0.828562	valid_0's binary_logloss: 0.533255
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[21]	valid_0's auc: 0.81442	valid_0's binary_logloss: 0.60553
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[228]	valid_0's auc: 0.803812	valid_0's binary_logloss: 0.518517
Controlled High-Capacity LGBM OOF AUC: 0.79767


In [ ]:
import pandas as pd
import numpy as np
import warnings
from scipy.optimize import minimize
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 1. Load Data
print("Loading datasets...")
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. Enhanced Feature Engineering with Clinical Age Brackets
def engineer_features(train_df, test_df):
    combined = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    kmeans = KMeans(n_clusters=12, random_state=42).fit(combined[['latitude', 'longitude']])
    combined['geo_cluster'] = kmeans.predict(combined[['latitude', 'longitude']])

    train_feat = combined.iloc[:len(train_df)].copy()
    test_feat  = combined.iloc[len(train_df):].copy()

    def process(df):
        df = df.copy()
        df['deathdate'] = pd.to_datetime(df['deathdate'])
        df['year']  = df['deathdate'].dt.year
        df['month'] = df['deathdate'].dt.month

        # Core non-linear features
        df['log_age']       = np.log1p(df['age'])

        # Clinical Age Brackets & Threshold Flags
        df['age_group'] = pd.cut(
            df['age'],
            bins=[-1, 1, 5, 18, 40, 60, 120],
            labels=[0, 1, 2, 3, 4, 5]
        ).astype(int)

        df['infant_heat']     = ((df['age'] <= 1) & (df['hot_days_30d'] > 0)).astype(int)
        df['elderly_heat']    = ((df['age'] >= 60) & (df['hot_days_30d'] > 0)).astype(int)
        df['is_infant']       = (df['age'] <= 2).astype(int)
        df['is_elderly']      = (df['age'] >= 60).astype(int)

        # Climate & Environmental Interactions
        df['temp_range']    = df['max_temperature'] - df['min_temperature']
        df['heat_stress']   = df['hot_days_30d'] * df['max_temperature']
        df['drought_idx']   = df['rain_sum_30d'] / (df['ndvi_30d'] + 1e-5)
        df['temp_ndvi']     = df['max_temperature'] / (df['ndvi_30d'] + 1e-5)

        year_sensitivity = {
            2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
            2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
            2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
        }
        df['year_sensitivity'] = df['year'].map(year_sensitivity)

        df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
        df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

        drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra']
        return df.drop(drop_cols, axis=1, errors='ignore')

    return process(train_feat), process(test_feat)

train_df, test_df = engineer_features(train, test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.drop('is_climate_sensitive', axis=1, errors='ignore').fillna(-999)

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))

print("\n--- Training Base Models with Clinical Age Features ---")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=6, num_leaves=40,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.7,
        subsample=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.6,
        subsample=0.8, gamma=0.1, reg_alpha=0.2, reg_lambda=1.5,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.015, depth=5,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=3.0,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]

    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

# 3. Numerical Optimization for Stacking Weights
def objective_function(weights):
    w1, w2, w3 = weights
    blend = (w1 * oof_lgbm) + (w2 * oof_xgb) + (w3 * oof_cat)
    return -roc_auc_score(y, blend)

res = minimize(objective_function, [0.33, 0.33, 0.33], bounds=[(0,1), (0,1), (0,1)], constraints={'type': 'eq', 'fun': lambda w: 1 - sum(w)})
w_lgbm, w_xgb, w_cat = res.x

print(f"\nOptimal Weights -> LGBM: {w_lgbm:.3f}, XGB: {w_xgb:.3f}, Cat: {w_cat:.3f}")

oof_ensemble = (w_lgbm * oof_lgbm) + (w_xgb * oof_xgb) + (w_cat * oof_cat)
test_ensemble = (w_lgbm * test_lgbm) + (w_xgb * test_xgb) + (w_cat * test_cat)

# 4. Threshold Optimization
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.001):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Clinical Age-Engineered Results ---")
print(f"Best Decision Threshold: {best_thresh:.3f}")
print(f"OOF Competition Score: {best_score:.5f}")
print(f"OOF ROC-AUC: {roc_auc_score(y, oof_ensemble):.5f}")

# 5. Save Submission
sub = ss.copy()
sub['TargetF1'] = (test_ensemble >= best_thresh).astype(int)
sub['TargetRAUC'] = test_ensemble
sub_filename = '/content/submission_clinical_age.csv'
sub.to_csv(sub_filename, index=False)
print(f"\nSaved '{sub_filename}' successfully!")

from google.colab import files
files.download(sub_filename)

Loading datasets...

--- Training Base Models with Clinical Age Features ---
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[47]	valid_0's auc: 0.829662	valid_0's binary_logloss: 0.544759
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[136]	valid_0's auc: 0.844326	valid_0's binary_logloss: 0.4875
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[14]	valid_0's auc: 0.83573	valid_0's binary_logloss: 0.606405
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[14]	valid_0's auc: 0.805262	valid_0's binary_logloss: 0.610133
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[79]	valid_0's auc: 0.805979	valid_0's binary_logloss: 0.533212

Optimal Weights -> LGBM: 0.333, XGB: 0.333, Cat: 0.333

--- Clinical Age-Engineered Results ---
Best Decision Threshold

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import warnings
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 1. Load Data
print("Loading datasets...")
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. Clean, High-Signal Feature Engineering
def engineer_features(train_df, test_df):
    combined = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    kmeans = KMeans(n_clusters=12, random_state=42).fit(combined[['latitude', 'longitude']])
    combined['geo_cluster'] = kmeans.predict(combined[['latitude', 'longitude']])

    train_feat = combined.iloc[:len(train_df)].copy()
    test_feat  = combined.iloc[len(train_df):].copy()

    def process(df):
        df = df.copy()
        df['deathdate'] = pd.to_datetime(df['deathdate'])
        df['year']  = df['deathdate'].dt.year
        df['month'] = df['deathdate'].dt.month

        # Core robust features
        df['log_age']       = np.log1p(df['age'])
        df['is_infant']     = (df['age'] <= 2).astype(int)
        df['is_elderly']    = (df['age'] >= 60).astype(int)
        df['temp_range']    = df['max_temperature'] - df['min_temperature']
        df['heat_stress']   = df['hot_days_30d'] * df['max_temperature']
        df['drought_idx']   = df['rain_sum_30d'] / (df['ndvi_30d'] + 1e-5)
        df['temp_ndvi']     = df['max_temperature'] / (df['ndvi_30d'] + 1e-5)

        year_sensitivity = {
            2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
            2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
            2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
        }
        df['year_sensitivity'] = df['year'].map(year_sensitivity)

        df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
        df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

        drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra']
        return df.drop(drop_cols, axis=1, errors='ignore')

    return process(train_feat), process(test_feat)

train_df, test_df = engineer_features(train, test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.drop('is_climate_sensitive', axis=1, errors='ignore').fillna(-999)

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

# Storage for Level-1 OOFs and Test predictions
oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))
oof_mlp  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))
test_mlp  = np.zeros(len(X_test))

print("\n--- Training 4 Diverse Base Models (LGBM, XGB, CatBoost, Neural Net) ---")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # 1. LightGBM
    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=6, num_leaves=40,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.7,
        subsample=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    # 2. XGBoost
    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.6,
        subsample=0.8, gamma=0.1, reg_alpha=0.2, reg_lambda=1.5,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    # 3. CatBoost
    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.015, depth=5,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=3.0,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    # 4. Neural Network (MLP) with Feature Scaling
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    mlp = MLPClassifier(
        hidden_layer_sizes=(64, 32), activation='relu', alpha=0.01,
        learning_rate_init=0.001, max_iter=300, random_state=42, early_stopping=True
    )
    mlp.fit(X_tr_scaled, y_tr)

    # Store Predictions
    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]
    oof_mlp[val_idx]  = mlp.predict_proba(X_val)[:, 1]

    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits
    test_mlp  += mlp.predict_proba(X_test)[:, 1] / folds.n_splits

# 3. Level-2 Meta-Learner Stacking
print("\n--- Training Level-2 Meta-Learner Stack ---")
X_meta_train = np.column_stack([oof_lgbm, oof_xgb, oof_cat, oof_mlp])
X_meta_test  = np.column_stack([test_lgbm, test_xgb, test_cat, test_mlp])

meta_model = LogisticRegression(C=1.0, random_state=42)
meta_model.fit(X_meta_train, y)

oof_ensemble = meta_model.predict_proba(X_meta_train)[:, 1]
test_ensemble = meta_model.predict_proba(X_meta_test)[:, 1]

print(f"Meta-Learner Coefficients (LGBM, XGB, Cat, MLP): {meta_model.coef_[0]}")

# 4. Threshold Optimization for Competition Score
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.001):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Meta-Stacking Results ---")
print(f"Best Decision Threshold: {best_thresh:.3f}")
print(f"Final OOF Competition Score: {best_score:.5f}")
print(f"Final OOF ROC-AUC: {roc_auc_score(y, oof_ensemble):.5f}")

# 5. Save Submission
sub = ss.copy()
sub['TargetF1'] = (test_ensemble >= best_thresh).astype(int)
sub['TargetRAUC'] = test_ensemble
sub_filename = '/content/submission_meta_learner.csv'
sub.to_csv(sub_filename, index=False)
print(f"\nSaved '{sub_filename}' successfully!")

from google.colab import files
files.download(sub_filename)

Loading datasets...

--- Training 4 Diverse Base Models (LGBM, XGB, CatBoost, Neural Net) ---
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[3]	valid_0's auc: 0.827422	valid_0's binary_logloss: 0.637627
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[109]	valid_0's auc: 0.844988	valid_0's binary_logloss: 0.495634
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[7]	valid_0's auc: 0.834974	valid_0's binary_logloss: 0.625295
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[11]	valid_0's auc: 0.805935	valid_0's binary_logloss: 0.619389
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[170]	valid_0's auc: 0.806468	valid_0's binary_logloss: 0.515582

--- Training Level-2 Meta-Learner Stack ---
Meta-Learner Coefficients (LGBM, XGB, Cat, MLP): [1.7718

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.1 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
import warnings
from scipy.optimize import minimize
from sklearn.neighbors import BallTree
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 1. Load Data
print("Loading datasets...")
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. Spatial BallTree Lag Feature Engineering + Core Features
def engineer_features_spatial(train_df, test_df):
    combined = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    # Geographic Clustering
    kmeans = KMeans(n_clusters=12, random_state=42).fit(combined[['latitude', 'longitude']])
    combined['geo_cluster'] = kmeans.predict(combined[['latitude', 'longitude']])

    # Spatial BallTree Nearest Neighbor Aggregations using Haversine (radians required)
    print("Computing spatial neighbor climate lags with BallTree...")
    coords_rad = np.radians(combined[['latitude', 'longitude']].values)
    balltree = BallTree(coords_rad, metric='haversine')

    # Find 5 nearest neighbors for each point (6 including self)
    distances, indices = balltree.query(coords_rad, k=6)

    # Extract neighbor averages for key environmental metrics
    env_cols = ['ndvi_30d', 'hot_days_30d', 'max_temperature', 'rain_sum_30d']
    for col in env_cols:
        neighbor_vals = combined[col].values[indices[:, 1:]] # exclude self at index 0
        combined[f'{col}_spatial_mean'] = neighbor_vals.mean(axis=1)
        combined[f'{col}_spatial_max'] = neighbor_vals.max(axis=1)

    train_feat = combined.iloc[:len(train_df)].copy()
    test_feat  = combined.iloc[len(train_df):].copy()

    def process(df):
        df = df.copy()
        df['deathdate'] = pd.to_datetime(df['deathdate'])
        df['year']  = df['deathdate'].dt.year
        df['month'] = df['deathdate'].dt.month

        # Core non-linear features
        df['log_age']       = np.log1p(df['age'])
        df['is_infant']     = (df['age'] <= 2).astype(int)
        df['is_elderly']    = (df['age'] >= 60).astype(int)
        df['temp_range']    = df['max_temperature'] - df['min_temperature']
        df['heat_stress']   = df['hot_days_30d'] * df['max_temperature']
        df['drought_idx']   = df['rain_sum_30d'] / (df['ndvi_30d'] + 1e-5)
        df['temp_ndvi']     = df['max_temperature'] / (df['ndvi_30d'] + 1e-5)

        year_sensitivity = {
            2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
            2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
            2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
        }
        df['year_sensitivity'] = df['year'].map(year_sensitivity)

        df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
        df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

        drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra']
        return df.drop(drop_cols, axis=1, errors='ignore')

    return process(train_feat), process(test_feat)

train_df, test_df = engineer_features_spatial(train, test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.drop('is_climate_sensitive', axis=1, errors='ignore').fillna(-999)

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))

print("\n--- Training Base Models with Spatial Lag Features ---")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=6, num_leaves=40,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.7,
        subsample=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.6,
        subsample=0.8, gamma=0.1, reg_alpha=0.2, reg_lambda=1.5,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.015, depth=5,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=3.0,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]

    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

# 3. Optimized Weight Blending
def objective_function(weights):
    w1, w2, w3 = weights
    blend = (w1 * oof_lgbm) + (w2 * oof_xgb) + (w3 * oof_cat)
    return -roc_auc_score(y, blend)

res = minimize(objective_function, [0.33, 0.33, 0.33], bounds=[(0,1), (0,1), (0,1)], constraints={'type': 'eq', 'fun': lambda w: 1 - sum(w)})
w_lgbm, w_xgb, w_cat = res.x

oof_ensemble = (w_lgbm * oof_lgbm) + (w_xgb * oof_xgb) + (w_cat * oof_cat)
test_ensemble = (w_lgbm * test_lgbm) + (w_xgb * test_xgb) + (w_cat * test_cat)

# 4. Probability Calibration via Isotonic Regression
ir = IsotonicRegression(out_of_bounds='clip')
ir.fit(oof_ensemble, y)
oof_calibrated = ir.transform(oof_ensemble)
test_calibrated = ir.transform(test_ensemble)

# 5. Threshold Optimization on Calibrated Probabilities
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.001):
    f1 = f1_score(y, (oof_calibrated >= thresh).astype(int))
    auc = roc_auc_score(y, oof_calibrated)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Spatial Lag + Calibration Results ---")
print(f"Best Decision Threshold: {best_thresh:.3f}")
print(f"Final OOF Competition Score: {best_score:.5f}")
print(f"Final OOF ROC-AUC: {roc_auc_score(y, oof_calibrated):.5f}")

# 6. Save Submission
sub = ss.copy()
sub['TargetF1'] = (test_calibrated >= best_thresh).astype(int)
sub['TargetRAUC'] = test_calibrated
sub_filename = '/content/submission_spatial_calibrated.csv'
sub.to_csv(sub_filename, index=False)
print(f"\nSaved '{sub_filename}' successfully!")

from google.colab import files
files.download(sub_filename)

Loading datasets...
Computing spatial neighbor climate lags with BallTree...

--- Training Base Models with Spatial Lag Features ---
[LightGBM] [Info] Number of positive: 1637, number of negative: 879
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002316 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5505
[LightGBM] [Info] Number of data points in the train set: 2516, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.650636 -> initscore=0.621836
[LightGBM] [Info] Start training from score 0.621836
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 200 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warni

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import warnings
from scipy.optimize import minimize
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 1. Load Data
print("Loading datasets...")
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. Exhaustive Age-Climate Crosses & Feature Engineering
def engineer_features_exhaustive(train_df, test_df):
    combined = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    kmeans = KMeans(n_clusters=12, random_state=42).fit(combined[['latitude', 'longitude']])
    combined['geo_cluster'] = kmeans.predict(combined[['latitude', 'longitude']])

    train_feat = combined.iloc[:len(train_df)].copy()
    test_feat  = combined.iloc[len(train_df):].copy()

    def process(df):
        df = df.copy()
        df['deathdate'] = pd.to_datetime(df['deathdate'])
        df['year']  = df['deathdate'].dt.year
        df['month'] = df['deathdate'].dt.month

        # Core non-linear features
        df['log_age']       = np.log1p(df['age'])
        df['is_infant']     = (df['age'] <= 2).astype(int)
        df['is_elderly']    = (df['age'] >= 60).astype(int)

        df['temp_range']    = df['max_temperature'] - df['min_temperature']
        df['heat_stress']   = df['hot_days_30d'] * df['max_temperature']
        df['drought_idx']   = df['rain_sum_30d'] / (df['ndvi_30d'] + 1e-5)
        df['temp_ndvi']     = df['max_temperature'] / (df['ndvi_30d'] + 1e-5)

        # Exhaustive Age-Climate Interactions (Targeting the dominant signal)
        df['age_x_temp']     = df['age'] * df['max_temperature']
        df['age_x_heatdays'] = df['age'] * df['hot_days_30d']
        df['age_x_ndvi']     = df['age'] * df['ndvi_30d']
        df['log_age_x_stress'] = df['log_age'] * df['heat_stress']
        df['age_div_rain']   = df['age'] / (df['rain_sum_30d'] + 1e-5)

        year_sensitivity = {
            2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
            2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
            2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
        }
        df['year_sensitivity'] = df['year'].map(year_sensitivity)

        df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
        df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

        drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra']
        return df.drop(drop_cols, axis=1, errors='ignore')

    return process(train_feat), process(test_feat)

train_df, test_df = engineer_features_exhaustive(train, test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.drop('is_climate_sensitive', axis=1, errors='ignore').fillna(-999)

neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

# Multi-Seed Blending Loop Configuration
SEEDS = [42, 1337, 2026]

oof_lgbm_total = np.zeros(len(X))
oof_xgb_total  = np.zeros(len(X))
oof_cat_total  = np.zeros(len(X))

test_lgbm_total = np.zeros(len(X_test))
test_xgb_total  = np.zeros(len(X_test))
test_cat_total  = np.zeros(len(X_test))

print(f"\n--- Running Multi-Seed Training Across Seeds: {SEEDS} ---")
for seed in SEEDS:
    print(f"\nTraining with Seed {seed}...")
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        lgbm = LGBMClassifier(
            n_estimators=3000, learning_rate=0.01, max_depth=6, num_leaves=40,
            scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.7,
            subsample=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=seed
        )
        lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
                 callbacks=[early_stopping(200), log_evaluation(0)])

        xgb = XGBClassifier(
            n_estimators=3000, learning_rate=0.01, max_depth=5,
            scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.6,
            subsample=0.8, gamma=0.1, reg_alpha=0.2, reg_lambda=1.5,
            random_state=seed, early_stopping_rounds=200, eval_metric='auc'
        )
        xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        cat = CatBoostClassifier(
            iterations=3000, learning_rate=0.015, depth=5,
            scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=3.0,
            random_seed=seed, early_stopping_rounds=200, verbose=False
        )
        cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

        oof_lgbm_total[val_idx] += lgbm.predict_proba(X_val)[:, 1] / len(SEEDS)
        oof_xgb_total[val_idx]  += xgb.predict_proba(X_val)[:, 1] / len(SEEDS)
        oof_cat_total[val_idx]  += cat.predict_proba(X_val)[:, 1] / len(SEEDS)

        test_lgbm_total += lgbm.predict_proba(X_test)[:, 1] / (len(SEEDS) * folds.n_splits)
        test_xgb_total  += xgb.predict_proba(X_test)[:, 1] / (len(SEEDS) * folds.n_splits)
        test_cat_total  += cat.predict_proba(X_test)[:, 1] / (len(SEEDS) * folds.n_splits)

# 3. Optimized Weight Blending
def objective_function(weights):
    w1, w2, w3 = weights
    blend = (w1 * oof_lgbm_total) + (w2 * oof_xgb_total) + (w3 * oof_cat_total)
    return -roc_auc_score(y, blend)

res = minimize(objective_function, [0.33, 0.33, 0.33], bounds=[(0,1), (0,1), (0,1)], constraints={'type': 'eq', 'fun': lambda w: 1 - sum(w)})
w_lgbm, w_xgb, w_cat = res.x

print(f"\nOptimal Multi-Seed Weights -> LGBM: {w_lgbm:.3f}, XGB: {w_xgb:.3f}, Cat: {w_cat:.3f}")

oof_ensemble = (w_lgbm * oof_lgbm_total) + (w_xgb * oof_xgb_total) + (w_cat * oof_cat_total)
test_ensemble = (w_lgbm * test_lgbm_total) + (w_xgb * test_xgb_total) + (w_cat * test_cat_total)

# 4. Threshold Optimization
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.001):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Multi-Seed + Exhaustive Feature Crosses Results ---")
print(f"Best Decision Threshold: {best_thresh:.3f}")
print(f"Final OOF Competition Score: {best_score:.5f}")
print(f"Final OOF ROC-AUC: {roc_auc_score(y, oof_ensemble):.5f}")

# 5. Save Submission
sub = ss.copy()
sub['TargetF1'] = (test_ensemble >= best_thresh).astype(int)
sub['TargetRAUC'] = test_ensemble
sub_filename = '/content/submission_multiseed_exhaustive.csv'
sub.to_csv(sub_filename, index=False)
print(f"\nSaved '{sub_filename}' successfully!")

from google.colab import files
files.download(sub_filename)

Loading datasets...

--- Running Multi-Seed Training Across Seeds: [42, 1337, 2026] ---

Training with Seed 42...
[LightGBM] [Info] Number of positive: 1637, number of negative: 879
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003877 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6019
[LightGBM] [Info] Number of data points in the train set: 2516, number of used features: 37
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.650636 -> initscore=0.621836
[LightGBM] [Info] Start training from score 0.621836
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 200 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli